In [ ]:
# ==========================================
# CELL 1: PURE INSTALLATION
# ==========================================
# We install the modern library and strictly prevent numpy from updating to 2.0
!pip install -q "numpy<2"
!pip install -q escnn

In [ ]:
# ==========================================
# CELL 1.2: Imports & Configuration
# ==========================================

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torchvision.datasets import PCAM
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score
import escnn.nn as gnn
import escnn.gspaces as gspaces

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
BATCH_SIZE = 32
EPOCHS = 5

Using device: cuda


In [ ]:
# ==========================================
# CELL 1.5: DRIVE MOUNT & DATA SYNC
# ==========================================
from google.colab import drive
import os
import shutil

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define your permanent Drive path and local Colab path
drive_path = '/content/drive/MyDrive/Erdos_Project/PCAM_Data'
local_path = './data/pcam'
os.makedirs(local_path, exist_ok=True)

# 3. The 6 PCam files
files = [
    "camelyonpatch_level_2_split_train_x.h5.gz",
    "camelyonpatch_level_2_split_train_y.h5.gz",
    "camelyonpatch_level_2_split_test_x.h5.gz",
    "camelyonpatch_level_2_split_test_y.h5.gz",
    "camelyonpatch_level_2_split_valid_x.h5.gz",
    "camelyonpatch_level_2_split_valid_y.h5.gz"
]

print("Checking Data Sync...")
for f in files:
    drive_file = os.path.join(drive_path, f)
    local_file = os.path.join(local_path, f)

    # Copy from Drive to Local Colab SSD for fast training
    if not os.path.exists(local_file) and os.path.exists(drive_file):
        print(f"Copying {f} to fast local storage...")
        shutil.copy(drive_file, local_file)

print("\nData is synced to local Colab storage!")

Mounted at /content/drive
Checking Data Sync...
Copying camelyonpatch_level_2_split_train_x.h5.gz to fast local storage...
Copying camelyonpatch_level_2_split_train_y.h5.gz to fast local storage...
Copying camelyonpatch_level_2_split_test_x.h5.gz to fast local storage...
Copying camelyonpatch_level_2_split_test_y.h5.gz to fast local storage...
Copying camelyonpatch_level_2_split_valid_x.h5.gz to fast local storage...
Copying camelyonpatch_level_2_split_valid_y.h5.gz to fast local storage...

Data is synced to local Colab storage!


In [ ]:
# ==========================================
# CELL 1.7: EXTRACT COMPRESSED FILES
# ==========================================
print("Extracting .gz files into raw .h5 files...")
!gunzip -f ./data/pcam/*.gz
print("Extraction complete! PyTorch will now recognize the dataset.")

Extracting .gz files into raw .h5 files...
Extraction complete! PyTorch will now recognize the dataset.


In [ ]:
# ==========================================
# CELL 2: D4 DATA PIPELINE
# ==========================================
import torch
import torchvision.transforms as T
from torchvision.datasets import PCAM
from torch.utils.data import DataLoader

pcam_mean = [0.701, 0.538, 0.692]
pcam_std  = [0.235, 0.277, 0.213]
normalize = T.Normalize(mean=pcam_mean, std=pcam_std)

train_transform = T.Compose([
    T.ToTensor(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    normalize
])

def get_test_transform(angle, reflect=False):
    """
    Deterministically applies D4 actions (rotations + flips).
    Uses rot90 and flip to ensure zero interpolation loss.
    """
    k_map = {0: 0, 90: 1, 180: 2, 270: 3}
    k = k_map[angle]
    transforms_list = [T.ToTensor()]

    # 1. Apply Lossless Rotation
    if k != 0:
        transforms_list.append(T.Lambda(lambda x: torch.rot90(x, k=k, dims=[-2, -1])))

    # 2. Apply Reflection (Horizontal flip)
    if reflect:
        transforms_list.append(T.Lambda(lambda x: torch.flip(x, dims=[-1])))

    transforms_list.append(normalize)
    return T.Compose(transforms_list)

print("Building D4 Training and Test DataLoaders...")

# Training Loader
train_ds = PCAM(root='./data', split='train', download=False, transform=train_transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# Full 8-State Test Manifold
test_loaders = {}
for angle in [0, 90, 180, 270]:
    # Pure Rotations
    ds_rot = PCAM(root='./data', split='test', download=False, transform=get_test_transform(angle, reflect=False))
    test_loaders[f"{angle}_deg"] = DataLoader(ds_rot, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # Reflections
    ds_ref = PCAM(root='./data', split='test', download=False, transform=get_test_transform(angle, reflect=True))
    test_loaders[f"{angle}_deg_flip"] = DataLoader(ds_ref, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"✅ Data Pipeline Ready. Auditing {len(test_loaders)} geometric states.")

Building D4 Training and Test DataLoaders...
✅ Data Pipeline Ready. Auditing 8 geometric states.


In [ ]:
# ==========================================
# CELL 3: ESCNN STEERABLE ARCHITECTURE
# ==========================================
class SteerablePCamModel(nn.Module):
    def __init__(self):
        super(SteerablePCamModel, self).__init__()

        # 1. The Symmetry Space (D4) - using the modern escnn syntax
        self.r2_act = gspaces.flipRot2dOnR2(N=4)

        # 2. Input: 3 scalar color channels (Trivial Representation)
        self.in_type = gnn.FieldType(self.r2_act, 3 * [self.r2_act.trivial_repr])

        # 3. Hidden Layers: Regular Representations to track orientation
        self.out_type_1 = gnn.FieldType(self.r2_act, 16 * [self.r2_act.regular_repr])
        self.out_type_2 = gnn.FieldType(self.r2_act, 32 * [self.r2_act.regular_repr])
        self.out_type_3 = gnn.FieldType(self.r2_act, 64 * [self.r2_act.regular_repr])

        # 4. Build the Steerable Blocks using continuous harmonics
        self.block1 = gnn.SequentialModule(
            gnn.R2Conv(self.in_type, self.out_type_1, kernel_size=5, padding=2, bias=False),
            gnn.InnerBatchNorm(self.out_type_1),
            gnn.ReLU(self.out_type_1, inplace=True),
            gnn.PointwiseMaxPool(self.out_type_1, 2)
        )

        self.block2 = gnn.SequentialModule(
            gnn.R2Conv(self.out_type_1, self.out_type_2, kernel_size=5, padding=2, bias=False),
            gnn.InnerBatchNorm(self.out_type_2),
            gnn.ReLU(self.out_type_2, inplace=True),
            gnn.PointwiseMaxPool(self.out_type_2, 2)
        )

        self.block3 = gnn.SequentialModule(
            gnn.R2Conv(self.out_type_2, self.out_type_3, kernel_size=5, padding=2, bias=False),
            gnn.InnerBatchNorm(self.out_type_3),
            gnn.ReLU(self.out_type_3, inplace=True),
            gnn.PointwiseMaxPool(self.out_type_3, 2)
        )

        # 5. Group Pooling: Integrates over the group to achieve pure invariance
        self.gpool = gnn.GroupPooling(self.out_type_3)

        # 6. Final Classifier
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 2) # Maps the 64 invariant channels to our 2 classes
        )

    def forward(self, x):
        # Lift standard tensor into the geometric space
        x = gnn.GeometricTensor(x, self.in_type)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.gpool(x)

        # Project back to a standard PyTorch tensor for the classifier
        x = x.tensor
        x = self.classifier(x)
        return x

# Initialize and move to GPU
escnn_model = SteerablePCamModel().to(device)
print("Steerable Architecture Initialized Successfully!")

Steerable Architecture Initialized Successfully!


In [ ]:
# ==========================================
# CELL 4: MODEL TRAINING LOOP
# ==========================================
def train_equivariant_model(model, loader, epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # DYNAMIC LEARNING RATE: Halves if the loss plateaus
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        loop = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")

        for batch_idx, (images, labels) in enumerate(loop):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # Track Average Loss
            running_loss += loss.item()
            avg_loss = running_loss / (batch_idx + 1)

            # Update progress bar
            loop.set_postfix(
                avg_loss=f"{avg_loss:.4f}",
                lr=f"{optimizer.param_groups[0]['lr']:.6e}"
            )

        # Step for the scheduler at the end of the epoch based on the average loss
        scheduler.step(avg_loss)

    return model

print("Starting 5-Epoch Training Run on Full PCam Dataset...")
escnn_model = train_equivariant_model(escnn_model, train_loader, EPOCHS)
print("Training Complete!")

Starting 5-Epoch Training Run on Full PCam Dataset...


Epoch 5/5: 100%|██████████| 8192/8192 [10:15<00:00, 13.32it/s, avg_loss=0.1389, lr=1.000000e-03]

Training Complete!


In [ ]:
# ==========================================
# CELL 5: AUTO-SAVE TO GOOGLE DRIVE
# ==========================================
import os
from datetime import datetime
import torch

# 1. Define your permanent Drive path
drive_model_dir = '/content/drive/MyDrive/Erdos_Project/Models'
os.makedirs(drive_model_dir, exist_ok=True)

# 2. Create a unique filename for this continuous steerable model
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
model_filename = f"escnn_D4_PCAM_5Epochs_{timestamp}.pth"
save_path = os.path.join(drive_model_dir, model_filename)

# 3. Save the actual learned weights and biases
print(f"Saving Steerable model weights to: {save_path} ...")
torch.save(escnn_model.state_dict(), save_path)

print("\n✅ Model successfully saved to Google Drive!")
print("Your trained network is now permanently secure.")

Saving Steerable model weights to: /content/drive/MyDrive/Erdos_Project/Models/escnn_D4_PCAM_5Epochs_20260317_0454.pth ...

✅ Model successfully saved to Google Drive!
Your trained network is now permanently secure.


In [ ]:
# ==========================================
# CELL 6: RELOAD ESCNN WEIGHTS
# ==========================================
import os
import torch

# 1. Define the exact path to your saved model
drive_model_dir = '/content/drive/MyDrive/Erdos_Project/Models'
model_filename = 'escnn_D4_PCAM_5Epochs_20260317_0454.pth'
load_path = os.path.join(drive_model_dir, model_filename)

print(f"Loading continuous harmonic weights from: {load_path}")

# 2. Load the weights with strict=False to ignore the missing temporary filter caches
escnn_model.load_state_dict(torch.load(load_path, map_location=device), strict=False)

# 3. CRITICAL: Set strictly to evaluation mode
escnn_model.eval()
print("✅ Model weights successfully restored! Geometric filters are ready.")

Loading continuous harmonic weights from: /content/drive/MyDrive/Erdos_Project/Models/escnn_D4_PCAM_5Epochs_20260317_0454.pth
✅ Model weights successfully restored! Geometric filters are ready.


In [ ]:
# ==========================================
# CELL 7: D4 MANIFOLD ROBUSTNESS AUDIT
# ==========================================
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm import tqdm

print("Running Steerable Evaluation across full D4 manifold...")
all_labels, all_probs, all_preds = {}, {}, {}
results = {}
base_predictions = None

escnn_model.eval()
with torch.no_grad():
    for state_name, loader in test_loaders.items():
        l_list, pr_list, pd_list = [], [], []
        loop = tqdm(loader, desc=f"Testing {state_name}", leave=False)

        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)
            outputs = escnn_model(images)

            # Extract probabilities for the 'Cancer' class
            probs = F.softmax(outputs, dim=1)[:, 1]
            _, preds = torch.max(outputs, 1)

            l_list.extend(labels.cpu().numpy())
            pr_list.extend(probs.cpu().numpy())
            pd_list.extend(preds.cpu().numpy())

        all_labels[state_name] = np.array(l_list)
        all_probs[state_name] = np.array(pr_list)
        all_preds[state_name] = np.array(pd_list)

        # Calculate Flip Rate against the "0_deg" anchor
        if state_name == "0_deg":
            base_predictions = all_preds[state_name]
            flip_rate = 0.0
        else:
            flips = np.sum(all_preds[state_name] != base_predictions)
            flip_rate = (flips / len(base_predictions)) * 100

        results[state_name] = {
            "Accuracy": accuracy_score(l_list, pd_list),
            "AUC": roc_auc_score(l_list, pr_list),
            "Flip_Rate": flip_rate
        }

# Print Formatting
print(f"\n{'='*65}")
print(f"ESCNN ROBUSTNESS RESULTS (FULL D4 MANIFOLD)")
print(f"{'='*65}")
print(f"{'State':<15} | {'Accuracy':<10} | {'AUC':<10} | {'Flip Rate'}")
print("-" * 65)
for state, m in results.items():
    print(f"{state:<15} | {m['Accuracy']:.4f}   | {m['AUC']:.4f}   | {m['Flip_Rate']:.2f}%")

Running Steerable Evaluation across full D4 manifold...



ESCNN ROBUSTNESS RESULTS (FULL D4 MANIFOLD)
State           | Accuracy   | AUC        | Flip Rate
-----------------------------------------------------------------
0_deg           | 0.8455   | 0.9315   | 0.00%
0_deg_flip      | 0.8455   | 0.9315   | 0.00%
90_deg          | 0.8455   | 0.9315   | 0.00%
90_deg_flip     | 0.8455   | 0.9315   | 0.00%
180_deg         | 0.8455   | 0.9315   | 0.00%
180_deg_flip    | 0.8455   | 0.9315   | 0.00%
270_deg         | 0.8455   | 0.9315   | 0.00%
270_deg_flip    | 0.8455   | 0.9315   | 0.00%


In [ ]:
# =======================================
# CELL 8: RECALL & SPECIFICITY STABILITY
# =======================================
from sklearn.metrics import recall_score, confusion_matrix

clinical_threshold = 0.25

print(f"\n{'='*95}")
print(f"   CLINICAL METRICS (THRESHOLD = {clinical_threshold*100}%)")
print(f"{'='*95}")
print(f"{'State':<15} | {'Recall':<10} | {'Missed (FN)':<15} | {'False Alarms (FP)':<18} | {'Specificity'}")
print("-" * 95)

for state_name in test_loaders.keys():
    # Apply the 25.00% Safety Threshold
    custom_preds = (all_probs[state_name] >= clinical_threshold).astype(int)

    recall = recall_score(all_labels[state_name], custom_preds)
    tn, fp, fn, tp = confusion_matrix(all_labels[state_name], custom_preds).ravel()

    # Calculate Specificity (True Negative Rate)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print(f"{state_name:<15} | {recall:.4f}     | {fn:<15} | {fp:<18} | {specificity:.4f}")

print("=" * 95)


   CLINICAL METRICS (THRESHOLD = 25.0%)
State           | Recall     | Missed (FN)     | False Alarms (FP)  | Specificity
-----------------------------------------------------------------------------------------------
0_deg           | 0.8136     | 3053            | 1500               | 0.9085
0_deg_flip      | 0.8136     | 3053            | 1500               | 0.9085
90_deg          | 0.8136     | 3053            | 1500               | 0.9085
90_deg_flip     | 0.8136     | 3053            | 1500               | 0.9085
180_deg         | 0.8136     | 3053            | 1500               | 0.9085
180_deg_flip    | 0.8136     | 3053            | 1500               | 0.9085
270_deg         | 0.8136     | 3053            | 1500               | 0.9085
270_deg_flip    | 0.8136     | 3053            | 1500               | 0.9085


In [ ]:
# ==========================================
# CELL 11: ESCNN 95% RECALL CALIBRATION
# ==========================================
from sklearn.metrics import precision_recall_curve
import numpy as np

# 1. Grab the 0-degree baseline raw data
# We test baseline calibration on standard, un-rotated slides
base_labels = all_labels["0_deg"]
base_probs = all_probs["0_deg"]

# 2. Calculate precision and recall for all possible thresholds
precisions, recalls, thresholds = precision_recall_curve(base_labels, base_probs)

# 3. Find the exact threshold where Recall hits exactly 95%
target_recall = 0.85
idx = np.where(recalls >= target_recall)[0][-1]
optimal_threshold = thresholds[idx]

# 4. Print Formatting
print(f"\n{'='*70}")
print(f"   ESCNN CALIBRATION (TARGETING {target_recall*100}% RECALL)")
print(f"{'='*70}")
print(f"To guarantee {target_recall*100}% of cancers are caught on a standard slide (0-deg),")
print(f"your clinical threshold must be set to:  {optimal_threshold:.4f} ({(optimal_threshold*100):.2f}%)")
print("======================================================================")


   ESCNN CALIBRATION (TARGETING 85.0% RECALL)
To guarantee 85.0% of cancers are caught on a standard slide (0-deg),
your clinical threshold must be set to:  0.1545 (15.45%)
